# 03 — Baselines et modèles ML classiques

Ce notebook compare des modèles non deep learning : Dummy, KNN cosinus, SVM linéaire, SGD logistic rapide, RandomForest, ExtraTrees, HistGradientBoosting, AdaBoost.

Important : la LogisticRegression classique lente est volontairement remplacée par `SGDClassifier(loss='log_loss')`.

In [1]:
import sys
from pathlib import Path
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = c:\Users\yeyuy\Downloads\projet_ml_audio_classique_v7_structured\projet_ml_audio_classique_v7_structured


## Charger les features

In [3]:
import pandas as pd
from src.config import FEATURE_DIR
from src.pipeline_steps import step_compare_models
features = pd.read_parquet(FEATURE_DIR / 'train_audio_features.parquet')
print(features.shape)

(106647, 522)


## Cross-validation

`preset='fast'` est recommandé au début. Pour un run final plus complet, utiliser `preset='balanced'`.

In [4]:
PRESET = 'fast'  # 'fast' ou 'balanced'
N_SPLITS = 3
results, meta = step_compare_models(features, preset=PRESET, n_splits=N_SPLITS)
display(results)
print(meta)

,model,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,balanced_accuracy_mean,balanced_accuracy_std
5,hist_gradient_boosting,0.734872,0.009089,0.767853,0.004472,0.694628,0.010214
4,extra_trees,0.694365,0.003880,0.724704,0.001953,0.780560,0.006788
1,cosine_knn,0.657985,0.008834,0.653321,0.002743,0.638237,0.010819
3,sgd_logistic_fast,0.341542,0.002257,0.352132,0.000518,0.409466,0.000970
2,linear_svm_balanced,0.340037,0.001759,0.353695,0.001688,0.410144,0.000781
0,dummy_most_frequent,0.000134,0.000000,0.000388,0.000000,0.004854,0.000000


{'n_samples': 105939, 'n_classes': 206, 'n_rare_classes_excluded': 0}


## Interprétation
Def:
- `dummy_most_frequent` donne le niveau zéro.
- `cosine_knn` vérifie si les features rapprochent bien les espèces similaires.
- `linear_svm_balanced` est souvent le meilleur compromis vitesse/score.
- `extra_trees` et `random_forest` testent les interactions non linéaires.
- `hist_gradient_boosting` teste un boosting plus rapide que GradientBoosting classique.

Les résultats de validation croisée montrent que les modèles basés sur les arbres de décision obtiennent les meilleures performances globales. HistGradientBoosting atteint le meilleur compromis sur les scores F1 avec macro_f1=0.735 et weighted_f1=0.768 tandis que ExtraTrees présente la meilleure balanced accuracy=0.781 indiquant une meilleure prise en compte des classes minoritaires.

À l’inverse, les modèles linéaires (LinearSVC, SGDClassifier) restent nettement moins performants malgré leur rapidité d’entraînement. Cela suggère que les relations entre les descripteurs acoustiques et les classes ne sont pas purement linéaires.

cosine_knn obtient des résultats intermédiaires, ce qui confirme que les features extraites conservent une structure acoustique cohérente entre espèces similaires.

score quasi nul du modèle dummy_most_frequent confirme que les performances observées ne proviennent pas simplement du déséquilibre des classes mais bien de l’apprentissage des caractéristiques audio.